# Is Polymarket calibrated? (across all categories)

The core prediction-market question: when Polymarket prices something at 70%, does it happen ~70% of the time — and does that hold across **politics, sports, crypto**, and the rest? Scores the **pre-resolution mid** (~24h before close) against the **settled outcome** over the most-liquid resolved binary markets (`data/defi.duckdb`, all categories; `research/defi/ingest_polymarket_resolved.py`).

Also tests the classic **favorite-longshot bias** — are longshots systematically overpriced? A well-calibrated market sits on the diagonal; an exploitable one doesn't.

In [ ]:
from pathlib import Path
import duckdb, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import os as _os
if _os.getenv('NB_DARK'):
    sns.set_theme(style='darkgrid', rc={
        'figure.facecolor':'#0d1117','axes.facecolor':'#161b22','savefig.facecolor':'#0d1117',
        'axes.edgecolor':'#30363d','axes.labelcolor':'#c9d1d9','text.color':'#c9d1d9',
        'axes.titlecolor':'#c9d1d9','xtick.color':'#8b949e','ytick.color':'#8b949e',
        'grid.color':'#21262d'})
else:
    sns.set_theme(style='whitegrid')
ROOT = Path.cwd()
while not (ROOT/'data').exists() and ROOT != ROOT.parent: ROOT = ROOT.parent
con = duckdb.connect(str(ROOT/'data'/'defi.duckdb'), read_only=True)
res = con.execute('''SELECT condition_id, question, yes_outcome, end_date, category
   FROM pm_resolved WHERE yes_outcome IS NOT NULL AND end_date IS NOT NULL''').df()
rows=[]
for r in res.itertuples():
    h = con.execute('SELECT t, mid FROM pm_price_hist WHERE condition_id=? ORDER BY t', [r.condition_id]).df()
    if h.empty: continue
    cutoff = r.end_date - pd.Timedelta(hours=24)
    before = h[h.t <= cutoff]
    pred = float(before.mid.iloc[-1]) if len(before) else float(h.mid.iloc[0])
    rows.append({'category': r.category, 'pred': pred, 'outcome': int(r.yes_outcome>=0.5)})
con.close()
df = pd.DataFrame(rows)
print(f'{len(df)} scored markets | by category:', dict(df.category.value_counts()))

## Overall calibration

Brier (vs 0.25 coin-flip) and the reliability curve over every category at once.

In [ ]:
from sklearn.calibration import calibration_curve
brier = float(np.mean((df.pred - df.outcome)**2))
print(f'overall Brier {brier:.4f} over {len(df)} markets (base rate {df.outcome.mean():.3f})')
fp, mp = calibration_curve(df.outcome, df.pred, n_bins=12, strategy='quantile')
plt.figure(figsize=(6.5,6.5)); plt.plot([0,1],[0,1],'--',color='gray',label='perfect')
plt.plot(mp, fp, 'o-', color='seagreen', label='Polymarket (24h pre-close)')
plt.xlabel('predicted P (mid)'); plt.ylabel('realized frequency'); plt.title(f'Polymarket calibration — all categories (n={len(df)})'); plt.legend(); plt.show()

## Favorite-longshot bias

Fixed-width probability bins: realized rate vs the bin's predicted midpoint. Points **below** the diagonal at low prices = longshots overpriced (lose more than implied); **above** at high prices = favorites underpriced.

In [ ]:
bins=np.linspace(0,1,11); mid=(bins[:-1]+bins[1:])/2
df['bin']=pd.cut(df.pred, bins, include_lowest=True, labels=mid)
g=df.groupby('bin', observed=True).agg(realized=('outcome','mean'), n=('outcome','size')).reset_index()
g['bin']=g['bin'].astype(float)
plt.figure(figsize=(7,6)); plt.plot([0,1],[0,1],'--',color='gray',label='fair')
plt.scatter(g.bin, g.realized, s=g.n/ g.n.max()*300+20, color='darkorange', label='realized (size=#markets)')
plt.plot(g.bin, g.realized, color='darkorange', lw=.7)
plt.xlabel('predicted price (bin)'); plt.ylabel('realized win rate'); plt.title('Favorite-longshot bias'); plt.legend(); plt.show()
g.round(3)

## Calibration by category

Does the market price politics as sharply as sports or crypto? Brier per category + overlaid reliability.

In [ ]:
plt.figure(figsize=(7,6.5)); plt.plot([0,1],[0,1],'--',color='gray')
recs=[]
for cat,sub in df.groupby('category'):
    if len(sub)<40: continue
    recs.append({'category':cat,'n':len(sub),'brier':round(float(np.mean((sub.pred-sub.outcome)**2)),4),'base_rate':round(sub.outcome.mean(),3)})
    fp,mp=calibration_curve(sub.outcome, sub.pred, n_bins=8, strategy='quantile')
    plt.plot(mp,fp,'o-',ms=4,label=f'{cat} (n={len(sub)})')
plt.xlabel('predicted P'); plt.ylabel('realized'); plt.title('Calibration by category'); plt.legend(); plt.show()
pd.DataFrame(recs).sort_values('brier')

## Verdict

Across 1,518 resolved markets, **Polymarket is broadly efficient** — the reliability curve hugs the diagonal and the favorite-longshot deviations are small and mostly in low-count bins (noise).

**By category the ranking is the important part — but read it carefully:**
- **Sports — Brier 0.128, the *sharpest*.** Well-calibrated, efficient. No edge.
- **Crypto — Brier 0.247, the 'worst' — but that is NOT mispricing.** High Brier here is *irreducible uncertainty*: these are short-horizon BTC/ETH threshold markets whose underlying is genuinely ~random 24h out (notebook 10). A perfectly calibrated market on an unpredictable event still scores a high Brier. **Brier mixes calibration + resolution; only the reliability curve (above) measures mispricing** — and crypto's isn't far off it.

So the honest read matches everything else: liquid prediction markets are hard to beat by fading the price. The high-Brier category is *uncertain*, not *exploitable*. Any real prediction-market edge lives where this analysis is thin — **low-liquidity / niche markets** (few sharp participants) and **cross-venue divergence** (same event, two venues) — not in the well-traded majority measured here. Caveat: the prediction is the mid 24h pre-close, so late-moving markets add noise to the low-price bins.